# Predicting Data Pipeline Failures

## Problem Statement

Data pipelines in large-scale Spark environments often fail due to factors such as:
- High data volume
- Transformation complexity (joins)
- Data skew
- Resource pressure

This project builds a machine learning model to predict pipeline failures before execution, enabling proactive monitoring and intervention.

## Technologies
- PySpark 
- Databricks
- MLflow 

## Dataset
- 100,000 simulated pipeline runs
- Features: workload size, partitions, joins, skew, execution time


# Step 1 — Data Generation

We simulate 100,000 pipeline runs with realistic characteristics.
Failure is determined by a combination of data volume, complexity, and skew —
mimicking real-world Spark pipeline behavior.

In [0]:
import random
from pyspark.sql.functions import current_timestamp, monotonically_increasing_id

random.seed(42)

def generate_pipeline_runs(n=100000):
    data = []
    for _ in range(n):
        rows       = random.randint(1000, 100000)
        partitions = random.randint(10, 300)
        joins      = random.randint(1, 15)
        skew       = random.choice([0, 1])
        exec_time  = random.randint(100, 1500)

        # Realistic failure logic
        score = (
            0.00003 * rows +
            0.05 * joins +
            1.5 * skew +
            0.002 * exec_time
        )
        failed = 1 if score > 5 else 0

        data.append((rows, partitions, joins, skew, exec_time, failed))
    return data

columns = [
    "rows_processed",
    "num_partitions",
    "num_joins",
    "skew_flag",
    "execution_time",
    "failed"
]

raw_data = generate_pipeline_runs(100000)
df       = spark.createDataFrame(raw_data, columns)


df = (
    df
    .withColumn("pipeline_id", monotonically_increasing_id())
    .withColumn("run_timestamp", current_timestamp())
    .repartition(50)
)


In [0]:
# Data generation sanity check 0 To check the class balance

df.groupBy("failed").count().show()

## Step 2 — Feature Engineering

Spark ML requires all input features to be combined into a single vector column.
We use VectorAssembler to transform individual columns into a unified `features` vector.

### Features used:
| Feature | Description |
|---------|------------|
| rows_processed | Volume of data in the pipeline run |
| num_partitions | Level of Spark parallelism |
| num_joins | Number of join operations (complexity) |
| skew_flag | Whether data skew was detected (0/1) |
| execution_time | Total runtime in seconds |

### Target:
| Column | Description |
|--------|------------|
| failed | 1 = failure, 0 = success |


In [0]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "rows_processed",
    "num_partitions",
    "num_joins",
    "skew_flag",
    "execution_time"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_ml = assembler.transform(df).select("pipeline_id", "features", "failed")

In [0]:
df_ml.show(5, truncate=False)

## Step 3 — Train/Test Split

We split the dataset into:
- **80% training** — model learns patterns
- **20% testing** — model is evaluated on unseen data


In [0]:
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Train count:", train_df.count())
print("Test count:", test_df.count())


print("=== Train Distribution ===")
train_df.groupBy("failed").count().show()

print("=== Test Distribution ===")
test_df.groupBy("failed").count().show()

## Step 4 — Model Training

We train a Logistic Regression model for binary classification:
- 0 → pipeline success
- 1 → pipeline failure

Logistic Regression is chosen as a strong baseline model —
simple, interpretable, and effective for structured data.

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="failed",
    maxIter=10
)

model = lr.fit(train_df)


In [0]:
predictions = model.transform(test_df)

In [0]:
predictions.select(
    "pipeline_id",
    "failed",
    "prediction",
    "probability"
).show(10, truncate=False)

## Step 5 — Model Evaluation

We evaluate the model on the **test set only** (unseen data).

### Metrics used:
| Metric | What it measures |
|--------|-----------------|
| Accuracy | Overall correctness |
| AUC | How well model separates success vs failure |
| Precision | When model predicts failure, how often is it right? |
| Recall | Of all real failures, how many did the model catch? |

In [0]:
correct = predictions.filter("failed = prediction").count()
total = predictions.count()
accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="failed",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(predictions)

print(f"AUC: {auc:.4f}")

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Precision
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="failed",
    predictionCol="prediction",
    metricName="precisionByLabel",
    metricLabel=1.0
)
precision = precision_evaluator.evaluate(predictions)

# Recall
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="failed",
    predictionCol="prediction",
    metricName="recallByLabel",
    metricLabel=1.0
)
recall = recall_evaluator.evaluate(predictions)

print(f"Precision (failure class): {precision:.4f}")
print(f"Recall (failure class):    {recall:.4f}")